### A funny experiment on Use vs Mention

It checks if the embedding of the word cat used in the sentence "The cat sat on the mat." is different from the word cat used in the sentence "Cat is an English noun."

Also it checks how much the brackets for the word "cat" change the embeddings for cat. 

Very very unfinished. But a bit funny. 


In [ ]:
import torch
from transformers import BertTokenizer, BertModel
import numpy as np

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased', output_hidden_states=True)
model.eval()

def get_word_embedding(sentence, target_word, layer=-1, aggregation='mean'):
    """
    Extract contextual embedding for target_word in sentence.
    
    Args:
        layer: which layer (-1 = last, or 0-12)
        aggregation: 'mean', 'first', or 'last' for subword handling
    """
    # Tokenize
    inputs = tokenizer(sentence, return_tensors='pt')
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    
    # Find target word token indices (handle subwords)
    target_indices = []
    target_lower = target_word.lower()
    
    for i, token in enumerate(tokens):
        clean_token = token.replace('##', '')
        if clean_token == target_lower or target_lower.startswith(clean_token):
            target_indices.append(i)
            # Continue collecting subwords
            if token.startswith('##') or (i+1 < len(tokens) and tokens[i+1].startswith('##')):
                continue
            elif len(target_indices) > 0 and not tokens[i].startswith('##'):
                # Check if we've completed the word
                reconstructed = ''.join(t.replace('##', '') for t in tokens[target_indices[0]:i+1])
                if reconstructed == target_lower:
                    break
    
    # Get embeddings
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Select layer
    hidden_states = outputs.hidden_states  # tuple of (batch, seq, hidden)
    layer_output = hidden_states[layer][0]  # (seq, hidden)
    
    # Aggregate subword embeddings
    target_embeddings = layer_output[target_indices]
    
    if aggregation == 'mean':
        return target_embeddings.mean(dim=0).numpy()
    elif aggregation == 'first':
        return target_embeddings[0].numpy()
    
    return target_embeddings.mean(dim=0).numpy()

# Example usage
use_sent = "The cat sat on the mat."
mention_sent = "'Cat' is an English noun."
mention_control_sent = "'Hammer' is an English noun."

emb_use = get_word_embedding(use_sent, "cat", layer=8)
emb_mention = get_word_embedding(mention_sent, "cat", layer=8)
emb_mention_control = get_word_embedding(mention_control_sent, "hammer", layer=8)

# These should differ if BERT captures use-mention!
similarity = np.dot(emb_use, emb_mention) / (np.linalg.norm(emb_use) * np.linalg.norm(emb_mention))
similarity_control = np.dot(emb_use, emb_mention_control) / (np.linalg.norm(emb_use) * np.linalg.norm(emb_mention_control))
print(f"Same word, different context: {similarity:.3f}")
print(f"Different word, different context: {similarity_control:.3f}")

Same word, different context: 0.705
Different word, different context: 0.329


In [ ]:
emb_cat_use = get_word_embedding("The cat sat on the mat.", "cat", layer=8)
emb_cat_mention = get_word_embedding("'Cat' is an English noun.", "cat", layer=8)
emb_hammer_use = get_word_embedding("The hammer struck the nail.", "hammer", layer=8)
emb_hammer_mention = get_word_embedding("'Hammer' is an English noun.", "hammer", layer=8)

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim_use_use = cosine_sim(emb_cat_use, emb_hammer_use)           
sim_mention_mention = cosine_sim(emb_cat_mention, emb_hammer_mention)  

print(f"cat vs hammer (both USED): {sim_use_use:.3f}")
print(f"cat vs hammer (both MENTIONED): {sim_mention_mention:.3f}")
print(f"Difference: {sim_mention_mention - sim_use_use:.3f}")

cat vs hammer (both USED): 0.404
cat vs hammer (both MENTIONED): 0.585
Difference: 0.180


In [ ]:
emb_cat_mention = get_word_embedding("'Cat' is an English noun.", "cat", layer=8)
emb_cat_mention_no_comma = get_word_embedding("Cat is an English noun.", "cat", layer=8)

sim_comma_no_comma = cosine_sim(emb_cat_mention, emb_cat_mention_no_comma) 

print(f"cat use vs cat mention (no comma): {sim_comma_no_comma:.3f}")

cat use vs cat mention (no comma): 0.931


In [ ]:
self_similarity_debug = cosine_sim(emb_cat_mention, emb_cat_mention)
print(f"Self-similarity check (should be 1.0): {self_similarity_debug:.3f}")

Self-similarity check (should be 1.0): 1.000


In [ ]:
# Synonyms: should show NEGATIVE mention boost
emb_cat_use = get_word_embedding("The cat sat on the mat.", "cat", layer=8)
emb_cat_mention = get_word_embedding("'Cat' is an English noun.", "cat", layer=8)
emb_feline_use = get_word_embedding("The feline stretched lazily.", "feline", layer=8)
emb_feline_mention = get_word_embedding("'Feline' is an English noun.", "feline", layer=8)

sim_use_use = cosine_sim(emb_cat_use, emb_feline_use)
sim_mention_mention = cosine_sim(emb_cat_mention, emb_feline_mention)

print(f"cat vs feline (both USED): {sim_use_use:.3f}")
print(f"cat vs feline (both MENTIONED): {sim_mention_mention:.3f}")
print(f"Mention boost: {sim_mention_mention - sim_use_use:+.3f}")


cat vs feline (both USED): 0.410
cat vs feline (both MENTIONED): 0.525
Mention boost: +0.114


In [ ]:
import torch
from transformers import BertTokenizer, BertModel
import numpy as np

class Experiment:
    def __init__(self, model_name='bert-base-uncased'):
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.model = BertModel.from_pretrained(model_name, output_hidden_states=True)
        self.model.eval()

    def get_robust_embedding(self, sentence, target_word_span, layer=-1):
        """
        Extracts embedding for a specific character span in the sentence.
        
        Args:
            sentence (str): The full sentence.
            target_word_span (tuple): (start_char, end_char) of the target word.
            layer (int): Layer index.
        """
        # Tokenize with offset mapping to link tokens back to characters
        inputs = self.tokenizer(sentence, return_tensors='pt', return_offsets_mapping=True)
        
        input_ids = inputs['input_ids']
        offsets = inputs['offset_mapping'][0] # (seq_len, 2)
        
        # Find tokens that fall within the character span of the target word
        target_token_indices = []
        start_char, end_char = target_word_span
        
        for idx, (token_start, token_end) in enumerate(offsets):
            # Skip special tokens [CLS], [SEP] which have (0,0) offset usually
            if token_start == token_end == 0: 
                continue
                
            # Check overlap: token is strictly inside or overlapping the target span
            if token_start >= start_char and token_end <= end_char:
                target_token_indices.append(idx)
        
        if not target_token_indices:
            raise ValueError(f"Could not align tokens for span {target_word_span} in '{sentence}'")

        # Run Model
        with torch.no_grad():
            outputs = self.model(input_ids)
        
        # Extract specific layer
        hidden_states = outputs.hidden_states
        layer_output = hidden_states[layer][0] # (seq_len, hidden_dim)
        
        # Stack and Mean Pool specific tokens
        target_vectors = layer_output[target_token_indices]
        final_embedding = torch.mean(target_vectors, dim=0).numpy()
        
        return final_embedding

# --- IMPROVED USAGE ---
experiment = Experiment()

# Define stimuli with explicit spans to avoid ambiguity
# "The cat sat..." -> "cat" is at indices 4 to 7
sent_use = "The cat sat on the mat."
span_use = (4, 7) 

# "'Cat' is an English noun." -> "Cat" is at indices 1 to 4 (ignoring quotes)
sent_mention = "'Cat' is an English noun."
span_mention = (1, 4)

emb_cat_use = experiment.get_robust_embedding(sent_use, span_use, layer=8)
emb_cat_mention = experiment.get_robust_embedding(sent_mention, span_mention, layer=8)

print(f"Extraction successful. Vector shape: {emb_cat_use.shape}")

NotImplementedError: return_offset_mapping is not available when using Python tokenizers. To use this feature, change your tokenizer to one deriving from transformers.PreTrainedTokenizerFast. More information on available tokenizers at https://github.com/huggingface/transformers/pull/2674